# Complemento Sesión 05 — SCD Type 2 con MERGE INTO

## ¿Qué es SCD Type 2?

Una **Slowly Changing Dimension Type 2** preserva el historial completo de cambios  
en una dimensión. Cuando un atributo cambia (ej. un cliente se muda de ciudad),  
no sobreescribimos el registro anterior — lo cerramos y creamos uno nuevo.

Cada versión del registro tiene:
- `valid_from` — fecha desde la que aplica esta versión
- `valid_to` — fecha hasta la que aplica (`9999-12-31` si es la versión activa)
- `is_current` — flag booleano para filtrar fácilmente la versión vigente
- `dim_sk` — surrogate key única por versión (no por cliente)

**Escenario de este ejemplo:**
- Cargamos la versión inicial de 3 clientes (enero 2024)
- Aplicamos un segundo batch donde CLI-001 cambió de ciudad y CLI-002 cambió de segmento
- CLI-003 no cambió — su registro no se toca

**Archivos necesarios:**
- `dim_clientes_v1.csv` — carga inicial
- `dim_clientes_v2.csv` — segundo batch con cambios

Subirlos a `/Volumes/dbassociate/default/vol_landing/session05/`

---
## 1. Setup

Importaciones y rutas. Si ya ejecutaste el laboratorio principal de la Sesión 05,  
las variables `CATALOG`, `SCHEMA` y `LAB_PATH` ya están definidas en el mismo cluster.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType, BooleanType
from pyspark.sql.functions import (
    col, lit, sha2, concat_ws, to_date, current_date, when
)

CATALOG  = "dbassociate"
SCHEMA   = "default"
LAB_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/vol_landing/session05"

print(f"Ruta: {LAB_PATH}")

---
## 2. Carga inicial: primera versión de la dimensión

Leemos el primer batch y construimos la tabla SCD2 desde cero.  
Cada registro recibe:
- `valid_from` = `fecha_inicio` del CSV
- `valid_to` = `9999-12-31` (registro activo, sin fecha de cierre)
- `is_current` = `true`
- `dim_sk` = hash SHA-256 sobre `(cliente_id, valid_from)` — único por versión

In [0]:
schema_cli = StructType([
    StructField("cliente_id",   StringType(), True),
    StructField("nombre",       StringType(), True),
    StructField("ciudad",       StringType(), True),
    StructField("segmento",     StringType(), True),
    StructField("fecha_inicio", StringType(), True),
])

clientes_v1 = (
    spark.read
    .option("header", "true")
    .schema(schema_cli)
    .csv(f"{LAB_PATH}/dim_clientes_v1.csv")
    .withColumn("valid_from",  to_date(col("fecha_inicio"), "yyyy-MM-dd"))
    .withColumn("valid_to",    to_date(lit("9999-12-31"),   "yyyy-MM-dd"))
    .withColumn("is_current",  lit(True))
    .withColumn(
        "dim_sk",
        sha2(concat_ws("|", col("cliente_id"), col("fecha_inicio")), 256)
    )
    .drop("fecha_inicio")
)

(
    clientes_v1.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.dim_clientes_scd2")
)

print("Carga inicial completada:")
display(spark.table(f"{CATALOG}.{SCHEMA}.dim_clientes_scd2"))

---
## 3. Segundo batch: detectar qué cambió

Antes de aplicar el MERGE, identificamos qué registros tienen atributos distintos  
respecto a la versión activa en la tabla.  

Comparamos `ciudad` y `segmento`. Si ambos coinciden, el registro no se toca.  
Este paso es importante: evita crear versiones fantasma para registros sin cambios.

In [0]:
clientes_v2 = (
    spark.read
    .option("header", "true")
    .schema(schema_cli)
    .csv(f"{LAB_PATH}/dim_clientes_v2.csv")
    .withColumn("valid_from_new", to_date(col("fecha_inicio"), "yyyy-MM-dd"))
    .drop("fecha_inicio")
)


# Unir con la versión activa actual para detectar diferencias
activos = (
    spark.table(f"{CATALOG}.{SCHEMA}.dim_clientes_scd2")
    .filter(col("is_current") == True)
    .select("cliente_id", "ciudad", "segmento")
    .withColumnRenamed("ciudad",   "ciudad_actual")
    .withColumnRenamed("segmento", "segmento_actual")
)


cambios = (
    clientes_v2
    .join(activos, on="cliente_id", how="left")
    .filter(
        (col("ciudad")   != col("ciudad_actual")) |
        (col("segmento") != col("segmento_actual"))
    )
    .drop("ciudad_actual", "segmento_actual")
)


print(f"Registros con cambios detectados: {cambios.count()}")
display(cambios)

---
## 4. MERGE INTO para SCD Type 2

El patrón SCD2 con MERGE requiere **dos operaciones por registro modificado**:

1. **Cerrar la versión anterior**: actualizar `valid_to = valid_from_new - 1 día` e `is_current = false`
2. **Insertar la versión nueva**: nuevo registro con `valid_from = fecha_del_cambio`, `valid_to = 9999-12-31`, `is_current = true`

Delta Lake soporta múltiples cláusulas `WHEN MATCHED` y `WHEN NOT MATCHED`,  
lo que permite expresar ambas operaciones en un solo MERGE.  

Primero cerramos los registros que cambiaron, luego insertamos las nuevas versiones.

In [0]:
# Paso 1: preparar las nuevas versiones como staging
nuevas_versiones = (
    cambios
    .withColumn("valid_from",  col("valid_from_new"))
    .withColumn("valid_to",    to_date(lit("9999-12-31"), "yyyy-MM-dd"))
    .withColumn("is_current",  lit(True))
    .withColumn(
        "dim_sk",
        sha2(concat_ws("|", col("cliente_id"), col("valid_from_new").cast("string")), 256)
    )
    .drop("valid_from_new")
)

# Escribimos los datos en una tabla temporal
(
    nuevas_versiones
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}._staging_scd2")
)


print(f"{CATALOG}.{SCHEMA}._staging_scd2 - Nuevas versiones a insertar:")
display(spark.table(f"{CATALOG}.{SCHEMA}._staging_scd2"))


In [0]:
# Paso 2: cerrar los registros activos que cambiaron
# Hacemos UPDATE sobre la tabla usando los cliente_id del staging
# valid_to = valid_from de la nueva versión - 1 día

spark.sql(f"""
MERGE INTO {CATALOG}.{SCHEMA}.dim_clientes_scd2 AS target
USING {CATALOG}.{SCHEMA}._staging_scd2 AS source
  ON target.cliente_id = source.cliente_id
 AND target.is_current = true
WHEN MATCHED THEN
  UPDATE SET
    target.valid_to    = dateadd(DAY, -1, source.valid_from),
    target.is_current  = false
""")

print("Registros anteriores cerrados.")
display(spark.table(f"{CATALOG}.{SCHEMA}.dim_clientes_scd2").orderBy("cliente_id", "valid_from"))

In [0]:
# Paso 3: insertar las nuevas versiones activas
# Las nuevas versiones no tienen match en la tabla (dim_sk es distinto)
# así que las insertamos directamente

spark.sql(f"""
MERGE INTO {CATALOG}.{SCHEMA}.dim_clientes_scd2 AS target
USING {CATALOG}.{SCHEMA}._staging_scd2 AS source
  ON target.dim_sk = source.dim_sk
WHEN NOT MATCHED THEN INSERT *
""")

# Limpieza de la tabla staging temporal
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}._staging_scd2")

In [0]:
display(spark.table(f"{CATALOG}.{SCHEMA}.dim_clientes_scd2").orderBy("cliente_id", "valid_from"))

---
## 5. Verificación: consultas sobre la dimensión SCD2

Una vez cargada la SCD2, mostramos los tres patrones de consulta más comunes:  
versión actual, historial completo de un cliente, y join point-in-time.

In [0]:
# Versión activa de cada cliente (la que se usa en reportes del día)
print("=== Versiones activas ===")
display(
    spark.table(f"{CATALOG}.{SCHEMA}.dim_clientes_scd2")
    .filter(col("is_current") == True)
    .select("cliente_id", "nombre", "ciudad", "segmento", "valid_from")
    .orderBy("cliente_id")
)

In [0]:
# Historial completo de CLI-001: muestra todas las versiones con sus fechas de vigencia
print("=== Historial CLI-001 ===")
display(
    spark.table(f"{CATALOG}.{SCHEMA}.dim_clientes_scd2")
    .filter(col("cliente_id") == "CLI-001")
    .select("cliente_id", "ciudad", "segmento", "valid_from", "valid_to", "is_current")
    .orderBy("valid_from")
)

In [0]:
# Consulta point-in-time: ¿cómo era CLI-001 el 2024-02-15?
# Útil para reprocesar hechos históricos con la dimensión vigente en ese momento

fecha_consulta = "2024-02-15"

print(f"=== CLI-001 al {fecha_consulta} ===")
display(
    spark.table(f"{CATALOG}.{SCHEMA}.dim_clientes_scd2")
    .filter(
        (col("cliente_id") == "CLI-001") &
        (col("valid_from") <= to_date(lit(fecha_consulta), "yyyy-MM-dd")) &
        (col("valid_to")   >= to_date(lit(fecha_consulta), "yyyy-MM-dd"))
    )
    .select("cliente_id", "ciudad", "segmento", "valid_from", "valid_to")
)

---
## Resumen

| Concepto | Implementación |
|---|---|
| Cierre de versión anterior | `MERGE INTO` con `UPDATE SET valid_to`, `is_current = false` |
| Nueva versión activa | `INSERT` separado con `valid_from = fecha_cambio`, `valid_to = 9999-12-31` |
| Surrogate key por versión | `sha2(cliente_id \| valid_from)` — distinto para cada versión del mismo cliente |
| Registro sin cambios | No se toca — el join de detección de cambios lo filtra |
| Consulta versión actual | `WHERE is_current = true` |
| Consulta point-in-time | `WHERE valid_from <= fecha AND valid_to >= fecha` |

**Por qué dos pasos y no uno solo:**  
Delta Lake no permite en un mismo MERGE hacer UPDATE sobre un registro  
e INSERT de uno nuevo para la misma clave de matching.  
Por eso el patrón estándar es: primero cerrar con MERGE, luego insertar con append.